<a href="https://colab.research.google.com/github/nenurd44/imbalanced-data-study/blob/main/TUNING_imbalanced_ma2_HPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
mlg_ulb_creditcardfraud_path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')

print('Data source import complete.')

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Data source import complete.


In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('/kaggle/input/creditcardfraud/creditcard.csv')

In [ ]:
def do_scaling (df, s):
    if s == "Robust":
        from sklearn.preprocessing import RobustScaler
        df['Amount'] = RobustScaler().fit_transform(df['Amount'].values.reshape(-1, 1))
        df['Time'] = RobustScaler().fit_transform(df['Time'].values.reshape(-1, 1))
    else:
        from sklearn.preprocessing import StandardScaler
        df['Amount'] = StandardScaler().fit_transform(df['Amount'].values.reshape(-1, 1))
        df['Time'] = StandardScaler().fit_transform(df['Time'].values.reshape(-1, 1))

In [ ]:
do_scaling(df, "Robust")
# do_scaling(df, "Standard")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,-0.994983,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,1.783274,0
1,-0.994983,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,-0.269825,0
2,-0.994972,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,4.983721,0
3,-0.994972,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,1.418291,0
4,-0.994960,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0.670579,0


In [ ]:
y = df['Class']
X = df.drop(columns=['Class'])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    'RF': RandomForestClassifier(),
    'XGB': XGBClassifier(
        random_state = 42,
        eval_metric = 'auc',
        use_label_encoder = False
    )
}

parameters = {
    'RF': {
        "n_estimators": [100, 200],
        "max_depth": [None, 5, 10],
        "min_samples_split": [2, 5],
        "class_weight": [None, "balanced"]
    },
    'XGB': {
        "n_estimators": [100, 300],
        "max_depth": [3, 7],
        "learning_rate": [0.01, 0.1],
        "scale_pos_weight": [1, 5, 10]
    }
}


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

grid = GridSearchCV(
    estimator = models['XGB'],
    param_grid = parameters['XGB'],
    scoring = 'average_precision',
    cv = 5,
    n_jobs = -1,
    verbose = 2
)

grid.fit(X_train, y_train)

print("Best parameters for XGB:", grid.best_params_)
print("Best score for XGB:", grid.best_score_)

# grid = GridSearchCV(
#     estimator = models['RF'],
#     param_grid = parameters['RF'],
#     scoring = 'average_precision',
#     cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
#     n_jobs = -1,
#     verbose = 2,
# )

# grid.fit(X_train, y_train)

# print("Best parameters for RF:", grid.best_params_)
# print("Best score for RF:", grid.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:12:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best parameters for XGB: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'scale_pos_weight': 5}
Best score for XGB: 0.8537469792250999
